# 05 Final Load Prep

**Project:** Smartwatch Health Analytics — User Risk Segmentation  
**Input:** `data/processed/clean_smartwatch_health_data.csv`  
**Output:** `data/processed/tableau_ready_dataset.csv`  
**Goal:** Compute all KPIs, add derived columns, and produce the final Tableau-ready extract.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()
DATA_PATH = PROJECT_ROOT / 'data/processed/clean_smartwatch_health_data.csv'
TABLEAU_READY_PATH = PROJECT_ROOT / 'data/processed/tableau_ready_dataset.csv'
KPI_SUMMARY_PATH = PROJECT_ROOT / 'data/processed/kpi_summary.csv'

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

## 1. Derived Columns

Adding computed fields to push the column count beyond 8 and enable rich Tableau filtering.

In [ ]:
# --- Derived Column 1: sleep_quality ---
# WHO recommends 7–9 hours for adults.
df['sleep_quality'] = pd.cut(
    df['sleep_duration_hours'],
    bins=[0, 5, 7, 9, 24],
    labels=['Very Poor (<5h)', 'Poor (5–7h)', 'Good (7–9h)', 'Excessive (>9h)'],
    right=False
).astype('object')
df['sleep_quality'] = df['sleep_quality'].where(df['sleep_duration_hours'].notna(), other=np.nan)

print('sleep_quality distribution:')
print(df['sleep_quality'].value_counts(dropna=False).to_string())

In [ ]:
# --- Derived Column 2: step_category ---
# WHO guidelines: 10,000 steps = active.
df['step_category'] = pd.cut(
    df['step_count'],
    bins=[0, 5000, 7500, 10000, 20000],
    labels=['Low (<5k)', 'Moderate (5k–7.5k)', 'Active (7.5k–10k)', 'High (>10k)'],
    right=False
).astype('object')
df['step_category'] = df['step_category'].where(df['step_count'].notna(), other=np.nan)

print('step_category distribution:')
print(df['step_category'].value_counts(dropna=False).to_string())

In [ ]:
# --- Derived Column 3: cardiovascular_risk_flag ---
# Elevated resting HR (> 100 BPM) = tachycardia risk.
df['cardiovascular_risk_flag'] = (df['heart_rate_bpm'] > 100).map({True: 'High Risk', False: 'Normal'})
df.loc[df['heart_rate_bpm'].isna(), 'cardiovascular_risk_flag'] = np.nan

print('cardiovascular_risk_flag distribution:')
print(df['cardiovascular_risk_flag'].value_counts(dropna=False).to_string())

In [ ]:
# --- Derived Column 4: blood_oxygen_status ---
# Clinical threshold: SpO2 < 95% = low.
df['blood_oxygen_status'] = pd.cut(
    df['blood_oxygen_level'],
    bins=[0, 90, 95, 101],
    labels=['Critical (<90%)', 'Low (90–95%)', 'Normal (>=95%)'],
    right=False
).astype('object')
df['blood_oxygen_status'] = df['blood_oxygen_status'].where(df['blood_oxygen_level'].notna(), other=np.nan)

print('blood_oxygen_status distribution:')
print(df['blood_oxygen_status'].value_counts(dropna=False).to_string())

In [ ]:
# --- Derived Column 5: stress_category ---
df['stress_category'] = df['stress_level'].map({
    1: 'Very Low', 2: 'Low', 3: 'Moderate', 4: 'High', 5: 'Very High'
})

print('stress_category distribution:')
print(df['stress_category'].value_counts(dropna=False).to_string())

In [ ]:
# --- Derived Column 6: wellness_score ---
# Composite score (0–100): lower stress + better sleep + more steps + normal HR + good SpO2.
# Normalize each component to 0–1 then weight.
def safe_norm(series, low, high, invert=False):
    clipped = series.clip(low, high)
    normed = (clipped - low) / (high - low)
    return (1 - normed) if invert else normed

sleep_score = safe_norm(df['sleep_duration_hours'], 3, 10) * 20     # 20 pts
step_score = safe_norm(df['step_count'], 0, 15000) * 25             # 25 pts
stress_score = safe_norm(df['stress_level'], 1, 5, invert=True) * 25  # 25 pts (lower = better)
hr_score = safe_norm(df['heart_rate_bpm'], 40, 120, invert=True) * 15  # 15 pts
spo2_score = safe_norm(df['blood_oxygen_level'], 88, 100) * 15     # 15 pts

df['wellness_score'] = (sleep_score + step_score + stress_score + hr_score + spo2_score).round(1)

print(f'Wellness Score — mean: {df["wellness_score"].mean():.1f}, median: {df["wellness_score"].median():.1f}')
print(f'Range: {df["wellness_score"].min():.1f} – {df["wellness_score"].max():.1f}')

## 2. KPI Computation

In [ ]:
# KPI 1: Average Resting Heart Rate by Activity Level
kpi_hr = df.groupby('activity_level')['heart_rate_bpm'].mean().round(2).reset_index()
kpi_hr.columns = ['activity_level', 'avg_heart_rate_bpm']
print('KPI 1 — Avg Resting Heart Rate by Activity Level:')
print(kpi_hr.to_string(index=False))

In [ ]:
# KPI 2: High Cardiovascular Risk Rate
total_valid = df['cardiovascular_risk_flag'].notna().sum()
high_risk_count = (df['cardiovascular_risk_flag'] == 'High Risk').sum()
high_risk_rate = round(high_risk_count / total_valid * 100, 2)
print(f'KPI 2 — Cardiovascular High Risk Rate: {high_risk_rate}% ({high_risk_count:,} users)')

In [ ]:
# KPI 3: Step Goal Achievement Rate (>= 10,000 steps)
step_valid = df['step_count'].notna().sum()
goal_achievers = (df['step_count'] >= 10000).sum()
step_goal_rate = round(goal_achievers / step_valid * 100, 2)
print(f'KPI 3 — Step Goal Achievement Rate: {step_goal_rate}% ({goal_achievers:,} users meet 10k steps/day)')

In [ ]:
# KPI 4: Poor Sleep Rate (< 7 hours)
sleep_valid = df['sleep_duration_hours'].notna().sum()
poor_sleepers = (df['sleep_duration_hours'] < 7).sum()
poor_sleep_rate = round(poor_sleepers / sleep_valid * 100, 2)
print(f'KPI 4 — Poor Sleep Rate: {poor_sleep_rate}% ({poor_sleepers:,} users sleep < 7h/night)')

In [ ]:
# KPI 5: Low Blood Oxygen Rate (< 95%)
spo2_valid = df['blood_oxygen_level'].notna().sum()
low_spo2 = (df['blood_oxygen_level'] < 95).sum()
low_spo2_rate = round(low_spo2 / spo2_valid * 100, 2)
print(f'KPI 5 — Low Blood Oxygen Rate: {low_spo2_rate}% ({low_spo2:,} users below 95% SpO₂)')

In [ ]:
# KPI 6: Average Wellness Score by Activity Level
kpi_wellness = df.groupby('activity_level')['wellness_score'].mean().round(1).reset_index()
kpi_wellness.columns = ['activity_level', 'avg_wellness_score']
print('KPI 6 — Avg Wellness Score by Activity Level:')
print(kpi_wellness.to_string(index=False))

In [ ]:
# Consolidated KPI Summary Table
kpi_summary = pd.DataFrame([
    {'KPI': 'High Cardiovascular Risk Rate (%)', 'Value': high_risk_rate, 'Unit': '%'},
    {'KPI': 'Step Goal Achievement Rate (%)', 'Value': step_goal_rate, 'Unit': '%'},
    {'KPI': 'Poor Sleep Rate (< 7h) (%)', 'Value': poor_sleep_rate, 'Unit': '%'},
    {'KPI': 'Low Blood Oxygen Rate (< 95%) (%)', 'Value': low_spo2_rate, 'Unit': '%'},
    {'KPI': 'Overall Avg Wellness Score', 'Value': df['wellness_score'].mean().round(1), 'Unit': '/100'},
])
print('=== KPI Summary ===')
print(kpi_summary.to_string(index=False))

## 3. Final Column Set — Tableau-Ready

In [ ]:
print(f'Final column count: {df.shape[1]}')
print('Columns:', df.columns.tolist())

## 4. Export

In [ ]:
TABLEAU_READY_PATH.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(TABLEAU_READY_PATH, index=False)
print(f'Tableau-ready dataset saved to: {TABLEAU_READY_PATH}')
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')

kpi_summary.to_csv(KPI_SUMMARY_PATH, index=False)
print(f'KPI summary saved to: {KPI_SUMMARY_PATH}')

## 5. Tableau Extract Summary

| Column | Type | Use in Tableau |
|---|---|---|
| user_id | Integer | Row identifier |
| heart_rate_bpm | Float | KPI, axis, filter |
| blood_oxygen_level | Float | KPI, axis |
| step_count | Integer | KPI, axis, filter |
| sleep_duration_hours | Float | KPI, axis |
| activity_level | String | Filter, color, groupby |
| stress_level | Integer | KPI, axis |
| sleep_quality | String | Filter, color |
| step_category | String | Filter, groupby |
| cardiovascular_risk_flag | String | Filter, highlight |
| blood_oxygen_status | String | Filter, highlight |
| stress_category | String | Filter, color |
| wellness_score | Float | KPI scorecard |

**Next Step:** Import `tableau_ready_dataset.csv` into Tableau Public to build the dashboard.